# Plot Blackrock Contents

This notebook allows you to load any Blackrock file (`.nsX` or `.nev`), list all available channels (including broadband, analog, and digital), and interactively plot across the entire duration of the recording.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import neo

# Set matplotlib to display inline or comment it out to be able to zoom in
%matplotlib qt

# --- CONFIG ---
# Replace with the actual path to your Blackrock file (.nsX or .nev)
BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20251203_NRR_RW011\Blackrock\NRR_RW011_008.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260116_NRR_RW012\Blackrock\NRR_RW012_019.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260119_NRR_RW013\Blackrock\NRR_RW013_010.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260130_NRR_RW014\Blackrock\NRR_RW014_010.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260202_NRR_RW015\Blackrock\NRR_RW015_010.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260206_NRR_RW016\Blackrock\NRR_RW016_020.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260218_NRR_RW017\Blackrock\NRR_RW017_013.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260220_NRR_RW018\Blackrock\NRR_RW018_005.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260226_NRR_RW019\Blackrock\NRR_RW019_028.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260311_NRR_RW022\Blackrock\NRR_RW022_023.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260326_NRR_RW026\Blackrock\NRR_RW026_010.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Ada\ANR_UA001\Blackrock\ANR_UA002.ns6"

### Step 1: Load File and List Channels

In [2]:
br_path = Path(BR_FILE_PATH)
if not br_path.exists() or not br_path.is_file():
    print(f"File not found: '{br_path}'")
    print("Please update the BR_FILE_PATH variable above.")
else:
    print(f"Loading Blackrock recording: {br_path.name}...")
    print("(This will automatically discover and load all aligned .nsX companion files!)")
    
    try:
        reader = neo.io.BlackrockIO(filename=str(br_path))
        bl = reader.read_block()
    except Exception as e:
        print(f"Failed to load Blackrock file(s): {e}")
        bl = None

    if bl is not None:
        # Gather all analog signals across all streams
        all_channels = []
        
        for seg in bl.segments:
            for asig in seg.analogsignals:
                fs = float(asig.sampling_rate.magnitude)
                dur = float(asig.t_stop - asig.t_start)
                
                # Neo includes names in the array annotations
                ch_names = asig.array_annotations.get('channel_names', [f"Ch{i}" for i in range(asig.shape[1])])
                
                for i in range(asig.shape[1]):
                    all_channels.append({
                        'name': ch_names[i],
                        'fs': fs,
                        'duration_sec': dur,
                        'signal': asig[:, i], # Reference to the neo object
                        'units': asig.units.dimensionality.string
                    })

        print("\n" + "="*40)
        print("       Recording Information       ")
        print("="*40)
        print(f"File:              {br_path.name}")
        print(f"Total Channels:    {len(all_channels)}")
        print("="*40)
        
        if len(all_channels) > 0:
            print("\nAvailable Channels (including auxiliary/analog):")
            for idx, ch_info in enumerate(all_channels):
                print(f"  [{idx:3d}] Name: {ch_info['name']:<15} | FS: {ch_info['fs']:>7.1f} Hz | Dur: {ch_info['duration_sec']:.1f}s")
        else:
            print("No channels found in this recording.")

Loading Blackrock recording: ANR_UA002.ns6...
(This will automatically discover and load all aligned .nsX companion files!)

       Recording Information       
File:              ANR_UA002.ns6
Total Channels:    143

Available Channels (including auxiliary/analog):
  [  0] Name: ghpos           | FS:  1000.0 Hz | Dur: 180.9s
  [  1] Name: gvpos           | FS:  1000.0 Hz | Dur: 180.9s
  [  2] Name: hhpos           | FS:  1000.0 Hz | Dur: 180.9s
  [  3] Name: hvpos           | FS:  1000.0 Hz | Dur: 180.9s
  [  4] Name: tcmd            | FS:  1000.0 Hz | Dur: 180.9s
  [  5] Name: vog_sync        | FS:  1000.0 Hz | Dur: 180.9s
  [  6] Name: heart_rate      | FS:  1000.0 Hz | Dur: 180.9s
  [  7] Name: hhv_filt        | FS:  1000.0 Hz | Dur: 180.9s
  [  8] Name: reward          | FS:  1000.0 Hz | Dur: 180.9s
  [  9] Name: hcmd1           | FS:  1000.0 Hz | Dur: 180.9s
  [ 10] Name: touchscreen_syn | FS:  1000.0 Hz | Dur: 180.9s
  [ 11] Name: pressure        | FS:  1000.0 Hz | Dur: 180.9s
 

### Step 2: Plot Selected Channel
Set the `CHANNEL_INDEX` variable to the index (0, 1, 2...) of the channel you want to plot from the list above.

In [3]:
# Set which channel to plot based on the index printed above
CHANNEL_INDEX = 100  # Example: 12 is 'unit'

if 'all_channels' not in locals() or len(all_channels) == 0:
    print("Please run the loading cell first to load the channels.")
elif not (0 <= CHANNEL_INDEX < len(all_channels)):
    print(f"Invalid CHANNEL_INDEX: {CHANNEL_INDEX}. Must be between 0 and {len(all_channels)-1}.")
else:
    selected_ch = all_channels[CHANNEL_INDEX]
    print(f"Selected channel: [{CHANNEL_INDEX}] {selected_ch['name']}")
    print(f"Extracting full {selected_ch['duration_sec']:g} seconds of data...")
    
    # Extract trace
    trace = selected_ch['signal'].magnitude.flatten()
    fs = selected_ch['fs']
    num_samples = len(trace)
    t_axis = np.arange(num_samples) / fs
    
    # Set up matplotlib for plotting
    fig, ax = plt.subplots(1, 1, figsize=(16, 5))
        
    fig.suptitle(f"Blackrock Entire Trace: {br_path.name} | Channel: {selected_ch['name']}", fontsize=14)

    # Plot
    ax.plot(t_axis, trace, color='#1f77b4', linewidth=0.8)
    units_str = selected_ch['units']
    if units_str == 'dimensionless':
        ax.set_ylabel("Amplitude")
    else:
        ax.set_ylabel(f"Amplitude ({units_str})")
        
    ax.grid(True, alpha=0.3)
    
    # Despine
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlabel("Time (seconds)")
    plt.tight_layout()
    plt.show()

Selected channel: [100] elec3-135
Extracting full 180.879 seconds of data...


In [6]:
# Plot selected Blackrock channel with a 300 Hz high-pass filter

from scipy.signal import butter, sosfiltfilt

# Set which channel to plot based on the index printed above
CHANNEL_INDEX = 60

# High-pass filter cutoff
HP_CUTOFF_HZ = 300.0
FILTER_ORDER = 4

if 'all_channels' not in locals() or len(all_channels) == 0:
    print("Please run the loading cell first to load the channels.")

elif not (0 <= CHANNEL_INDEX < len(all_channels)):
    print(f"Invalid CHANNEL_INDEX: {CHANNEL_INDEX}. Must be between 0 and {len(all_channels)-1}.")

else:
    selected_ch = all_channels[CHANNEL_INDEX]
    print(f"Selected channel: [{CHANNEL_INDEX}] {selected_ch['name']}")
    print(f"Extracting full {selected_ch['duration_sec']:g} seconds of data...")
    print(f"Applying {HP_CUTOFF_HZ:g} Hz high-pass filter...")

    # Extract trace
    trace = selected_ch['signal'].magnitude.flatten()
    fs = selected_ch['fs']
    num_samples = len(trace)
    t_axis = np.arange(num_samples) / fs

    # Check that cutoff is valid
    nyquist = fs / 2.0
    if HP_CUTOFF_HZ >= nyquist:
        raise ValueError(
            f"High-pass cutoff {HP_CUTOFF_HZ} Hz must be less than Nyquist frequency {nyquist} Hz."
        )

    # Design and apply high-pass Butterworth filter
    sos = butter(
        FILTER_ORDER,
        HP_CUTOFF_HZ,
        btype="highpass",
        fs=fs,
        output="sos"
    )

    # Zero-phase filtering
    filtered_trace = sosfiltfilt(sos, trace)

    # Set up matplotlib for plotting
    fig, ax = plt.subplots(1, 1, figsize=(16, 5))

    fig.suptitle(
        f"Blackrock Trace High-Pass Filtered at {HP_CUTOFF_HZ:g} Hz: "
        f"{br_path.name} | Channel: {selected_ch['name']}",
        fontsize=14
    )

    # Plot filtered trace
    ax.plot(t_axis, filtered_trace, color='#d62728', linewidth=0.8)

    units_str = selected_ch['units']
    if units_str == 'dimensionless':
        ax.set_ylabel("Amplitude")
    else:
        ax.set_ylabel(f"Amplitude ({units_str})")

    ax.grid(True, alpha=0.3)

    # Despine
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlabel("Time (seconds)")
    plt.tight_layout()
    plt.show()

Selected channel: [60] elec2-66
Extracting full 180.879 seconds of data...
Applying 300 Hz high-pass filter...


In [11]:
# ============================================================
# Blackrock/Neo all_channels -> UA spike detection + windowed plots
# Electrode inputs only: 15 through 142 inclusive
# ============================================================

import os
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
from matplotlib import cm

import spikeinterface as si
import spikeinterface.preprocessing as spre
from spikeinterface.core import NumpyRecording
from spikeinterface.sortingcomponents.peak_detection import detect_peaks


# -----------------------------
# User settings
# -----------------------------
ELECTRODE_CHANNEL_INDICES = list(range(15, 143))  # inclusive 15..142

CHANNEL_INDEX = 60          # original all_channels index to plot
HP_CUTOFF_HZ = 300.0

DETECT_THRESHOLD = 3.0
PEAK_SIGN = "neg"           # "neg", "pos", or "both"

MIN_AMPLITUDE_UV = None
MAX_AMPLITUDE_UV = 500.0
MIN_SNR = 2.5

BIN_MS = 20.0
HEATMAP_CMAP = "jet"
HEATMAP_VMIN = None
HEATMAP_VMAX = None

TEMPLATE_PATH = r"E:\NHP_Cerebellum_Project_2025_RCP_Analysis_Git\RCP_analysis\config\waveform_templates\median_extremum_templates_norm_UA_PortB.npy"
USE_MATCHED_FILTERING_IF_AVAILABLE = True

# Plot window.
# Example: 1 minute 10 seconds to 1 minute 15 seconds = 70.0 to 75.0 sec.
PLOT_WINDOW_SEC = (70.0, 75.0)

# If you want full recording heatmap/trace, set:
# PLOT_WINDOW_SEC = None

# Optional vertical reference line inside the plotted window.
# Set to None to disable. For this free-running window plot, usually None is fine.
REF_LINE_SEC = None
# REF_LINE_SEC = 70.0


# -----------------------------
# Helpers
# -----------------------------
def _as_float_array(x):
    """
    Convert Neo/quantities arrays or normal arrays to plain float ndarray.
    Assumes values are already in the units stored in all_channels.
    """
    try:
        # quantities.Quantity
        x = x.magnitude
    except Exception:
        pass
    return np.asarray(x, dtype=np.float32)


def _get_trace_units(ch_dict):
    return str(ch_dict.get("units", ""))


def _samples_from_window(window_sec, fs, n_samples):
    if window_sec is None:
        return 0, n_samples

    start_sec, end_sec = window_sec
    if start_sec is None:
        start_sec = 0.0
    if end_sec is None:
        end_sec = n_samples / fs

    start_frame = int(np.floor(float(start_sec) * fs))
    end_frame = int(np.ceil(float(end_sec) * fs))

    start_frame = max(0, min(start_frame, n_samples))
    end_frame = max(start_frame, min(end_frame, n_samples))

    return start_frame, end_frame


def filter_peaks_by_amplitude_snr(
    peaks,
    traces,
    noise_levels,
    min_amplitude_uv=None,
    max_amplitude_uv=500.0,
    min_snr=2.5,
    peak_sign="neg",
):
    """
    Production-style amplitude/SNR filter.

    Assumes traces are in the same numeric units as all_channels.
    If your Neo-loaded signals are already in uV, this directly matches uV thresholds.
    If they are in V or mV, convert before building recording or adjust thresholds.
    """
    if peaks is None or len(peaks) == 0:
        return peaks

    sample_inds = peaks["sample_index"].astype(np.int64)
    chan_inds = peaks["channel_index"].astype(np.int64)

    valid_bounds = (
        (sample_inds >= 0) &
        (sample_inds < traces.shape[0]) &
        (chan_inds >= 0) &
        (chan_inds < traces.shape[1])
    )

    if not np.all(valid_bounds):
        peaks = peaks[valid_bounds]
        sample_inds = sample_inds[valid_bounds]
        chan_inds = chan_inds[valid_bounds]

    if len(peaks) == 0:
        return peaks

    vals = traces[sample_inds, chan_inds]

    if peak_sign == "neg":
        amp = -vals
    elif peak_sign == "pos":
        amp = vals
    else:
        amp = np.abs(vals)

    nl = np.asarray(noise_levels, dtype=np.float32)
    snr = amp / np.maximum(nl[chan_inds], np.finfo(np.float32).eps)

    keep = np.ones(len(peaks), dtype=bool)

    if min_amplitude_uv is not None:
        keep &= amp >= float(min_amplitude_uv)

    if max_amplitude_uv is not None:
        keep &= amp <= float(max_amplitude_uv)

    if min_snr is not None:
        keep &= snr >= float(min_snr)

    return peaks[keep]


def make_spike_rate_heatmap(
    peaks,
    n_channels,
    fs,
    window_sec,
    bin_ms,
):
    """
    Returns:
      rate_hz: shape n_channels x n_bins
      x_edges_sec: shape n_bins + 1
    """
    bin_sec = float(bin_ms) / 1000.0

    if window_sec is None:
        if peaks is None or len(peaks) == 0:
            start_sec = 0.0
            end_sec = bin_sec
        else:
            start_sec = 0.0
            end_sec = float(np.max(peaks["sample_index"])) / fs
            end_sec = max(end_sec, bin_sec)
    else:
        start_sec, end_sec = map(float, window_sec)

    if end_sec <= start_sec:
        raise ValueError(f"Invalid window_sec={window_sec}. End must be greater than start.")

    n_bins = int(np.ceil((end_sec - start_sec) / bin_sec))
    x_edges_sec = start_sec + np.arange(n_bins + 1) * bin_sec

    counts = np.zeros((n_channels, n_bins), dtype=np.float32)

    if peaks is not None and len(peaks) > 0:
        peak_times_sec = peaks["sample_index"].astype(np.float64) / fs
        peak_ch = peaks["channel_index"].astype(np.int64)

        in_win = (
            (peak_times_sec >= start_sec) &
            (peak_times_sec < x_edges_sec[-1]) &
            (peak_ch >= 0) &
            (peak_ch < n_channels)
        )

        peak_times_sec = peak_times_sec[in_win]
        peak_ch = peak_ch[in_win]

        bin_idx = np.floor((peak_times_sec - start_sec) / bin_sec).astype(np.int64)
        valid = (bin_idx >= 0) & (bin_idx < n_bins)

        np.add.at(counts, (peak_ch[valid], bin_idx[valid]), 1.0)

    rate_hz = counts / bin_sec
    return rate_hz, x_edges_sec


# -----------------------------
# Validate and stack selected electrode channels
# -----------------------------
if "all_channels" not in globals():
    raise NameError("all_channels is not defined. Run your Neo Blackrock loading cell first.")

if CHANNEL_INDEX not in ELECTRODE_CHANNEL_INDICES:
    raise ValueError(
        f"CHANNEL_INDEX={CHANNEL_INDEX} is not in ELECTRODE_CHANNEL_INDICES "
        f"({ELECTRODE_CHANNEL_INDICES[0]}..{ELECTRODE_CHANNEL_INDICES[-1]})."
    )

fs_values = [float(all_channels[i]["fs"]) for i in ELECTRODE_CHANNEL_INDICES]
if len(set(fs_values)) != 1:
    raise ValueError(f"Selected electrode channels do not all have same fs: {sorted(set(fs_values))}")

fs = fs_values[0]
n_ch = len(ELECTRODE_CHANNEL_INDICES)

channel_names = [all_channels[i]["name"] for i in ELECTRODE_CHANNEL_INDICES]
channel_ids = [str(i) for i in ELECTRODE_CHANNEL_INDICES]

signals = [_as_float_array(all_channels[i]["signal"]) for i in ELECTRODE_CHANNEL_INDICES]
min_len = min(len(x) for x in signals)

if len(set(len(x) for x in signals)) != 1:
    print("Warning: selected channels have unequal lengths; truncating to shortest length.")

traces = np.column_stack([x[:min_len] for x in signals]).astype(np.float32, copy=False)

n_samples = traces.shape[0]
duration_sec = n_samples / fs

print(f"Using electrode inputs: {ELECTRODE_CHANNEL_INDICES[0]}..{ELECTRODE_CHANNEL_INDICES[-1]}")
print(f"n_channels = {n_ch}")
print(f"fs = {fs:.1f} Hz")
print(f"duration = {duration_sec:.3f} sec")
print(f"trace units from CHANNEL_INDEX {CHANNEL_INDEX}: {_get_trace_units(all_channels[CHANNEL_INDEX])}")
print(f"Plot window = {PLOT_WINDOW_SEC}")

plot_ch_local_index = ELECTRODE_CHANNEL_INDICES.index(CHANNEL_INDEX)
plot_ch_name = all_channels[CHANNEL_INDEX]["name"]

print(f"Plotting original all_channels[{CHANNEL_INDEX}] = {plot_ch_name}")
print(f"Local electrode-channel index = {plot_ch_local_index}")


# -----------------------------
# Build SpikeInterface recording
# -----------------------------
rec = NumpyRecording(
    traces_list=[traces],
    sampling_frequency=fs,
    channel_ids=channel_ids,
)

# Dummy locations so locally_exclusive/matched_filtering can operate.
# Production script used 400 um spacing.
locs = np.zeros((n_ch, 2), dtype=np.float32)
side = int(np.ceil(np.sqrt(n_ch)))
for i in range(n_ch):
    locs[i, 0] = (i % side) * 400.0
    locs[i, 1] = (i // side) * 400.0

rec.set_channel_locations(locs)


# -----------------------------
# High-pass filter
# -----------------------------
rec_hp = spre.highpass_filter(rec, freq_min=float(HP_CUTOFF_HZ))

# Materialize high-pass traces for plotting and amplitude/SNR filtering.
# For ~181 s x 128 ch x 30 kHz this can be memory-heavy but usually manageable as float32.
print("Materializing high-pass traces...")
traces_hp = rec_hp.get_traces(segment_index=0).astype(np.float32, copy=False)

print("Estimating noise levels by MAD...")
noise_levels = si.get_noise_levels(
    rec_hp,
    method="mad",
    return_in_uV=False,
    n_jobs=1,
)


# -----------------------------
# Peak detection: matched filtering if possible, else locally_exclusive
# -----------------------------
peaks = None
detection_method_used = None

if USE_MATCHED_FILTERING_IF_AVAILABLE and os.path.exists(TEMPLATE_PATH):
    try:
        ua_template = np.load(TEMPLATE_PATH).astype(np.float32).squeeze()

        if ua_template.ndim != 1:
            raise ValueError(f"Template must be 1D after squeeze; got shape {ua_template.shape}")

        ms_before = (np.argmin(ua_template) / fs) * 1000.0

        print(f"Using matched_filtering template:")
        print(f"  {TEMPLATE_PATH}")
        print(f"  template shape = {ua_template.shape}")
        print(f"  ms_before = {ms_before:.4f}")

        peaks = detect_peaks(
            rec_hp,
            method="matched_filtering",
            detect_threshold=float(DETECT_THRESHOLD),
            peak_sign=PEAK_SIGN,
            ms_before=ms_before,
            prototype=ua_template,
            radius_um=0,
            n_jobs=1,
        )

        detection_method_used = "matched_filtering"

    except Exception as e:
        print("Matched filtering failed; falling back to locally_exclusive.")
        print(f"Reason: {repr(e)}")
        peaks = None

elif USE_MATCHED_FILTERING_IF_AVAILABLE:
    print("Template path does not exist; falling back to locally_exclusive:")
    print(f"  {TEMPLATE_PATH}")

if peaks is None:
    print("Using locally_exclusive peak detection.")
    peaks = detect_peaks(
        rec_hp,
        method="locally_exclusive",
        detect_threshold=float(DETECT_THRESHOLD),
        peak_sign=PEAK_SIGN,
        noise_levels=noise_levels,
        n_jobs=1,
    )
    detection_method_used = "locally_exclusive"

print(f"Detected peaks before amplitude/SNR filtering: {len(peaks):,}")
print(f"Detection method used: {detection_method_used}")


# -----------------------------
# Amplitude/SNR filtering
# -----------------------------
peaks_filt = filter_peaks_by_amplitude_snr(
    peaks=peaks,
    traces=traces_hp,
    noise_levels=noise_levels,
    min_amplitude_uv=MIN_AMPLITUDE_UV,
    max_amplitude_uv=MAX_AMPLITUDE_UV,
    min_snr=MIN_SNR,
    peak_sign=PEAK_SIGN,
)

print(f"Detected peaks after amplitude/SNR filtering:  {len(peaks_filt):,}")

if len(peaks_filt) > 0:
    rates_by_ch = np.bincount(peaks_filt["channel_index"], minlength=n_ch) / duration_sec
    print(f"Mean channel rate after filtering: {np.mean(rates_by_ch):.2f} Hz")
    print(f"Median channel rate after filtering: {np.median(rates_by_ch):.2f} Hz")
    print(f"Max channel rate after filtering: {np.max(rates_by_ch):.2f} Hz")


# -----------------------------
# Windowed trace plot with spike dots
# -----------------------------
start_frame, end_frame = _samples_from_window(PLOT_WINDOW_SEC, fs, n_samples)
t_sec = np.arange(start_frame, end_frame, dtype=np.float64) / fs
trace_win = traces_hp[start_frame:end_frame, plot_ch_local_index]

ch_peaks = peaks_filt[peaks_filt["channel_index"] == plot_ch_local_index]
ch_peak_samples = ch_peaks["sample_index"].astype(np.int64)

in_trace_win = (ch_peak_samples >= start_frame) & (ch_peak_samples < end_frame)
ch_peak_samples_win = ch_peak_samples[in_trace_win]
ch_peak_times_win = ch_peak_samples_win / fs
ch_peak_vals_win = traces_hp[ch_peak_samples_win, plot_ch_local_index]

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(t_sec, trace_win, color="black", linewidth=0.7)

ax.scatter(
    ch_peak_times_win,
    ch_peak_vals_win,
    s=22,
    color="red",
    edgecolor="none",
    zorder=5,
    label=f"Detected spikes n={len(ch_peak_samples_win)}",
)

if REF_LINE_SEC is not None:
    ax.axvline(float(REF_LINE_SEC), color="Red", alpha=0.8, linewidth=1.2, ls="--")

ax.set_title(
    f"High-pass {HP_CUTOFF_HZ:g} Hz trace with detected spikes\n"
    f"all_channels[{CHANNEL_INDEX}] {plot_ch_name} | "
    f"threshold={DETECT_THRESHOLD:g}, sign={PEAK_SIGN}, method={detection_method_used}"
)
ax.set_xlabel("Time (s)")
ax.set_ylabel(f"High-pass signal ({_get_trace_units(all_channels[CHANNEL_INDEX])})")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right", frameon=False)

plt.tight_layout()
plt.show()


# -----------------------------
# Windowed all-channel heatmap using pcolormesh style
# -----------------------------
heatmap_rate_hz, x_edges_sec = make_spike_rate_heatmap(
    peaks=peaks_filt,
    n_channels=n_ch,
    fs=fs,
    window_sec=PLOT_WINDOW_SEC,
    bin_ms=BIN_MS,
)

heatmap_masked = ma.masked_invalid(heatmap_rate_hz)

cmap_local = cm.get_cmap(HEATMAP_CMAP).copy()
cmap_local.set_bad(color="gray")

y_edges = np.arange(n_ch + 1)

fig, ax = plt.subplots(figsize=(14, 7))

im = ax.pcolormesh(
    x_edges_sec,
    y_edges,
    heatmap_masked,
    cmap=cmap_local,
    vmin=HEATMAP_VMIN,
    vmax=HEATMAP_VMAX,
    shading="auto",
    antialiased=False,
    rasterized=True,
)
im.set_edgecolor("none")

if REF_LINE_SEC is not None:
    ax.axvline(float(REF_LINE_SEC), color="Red", alpha=0.8, linewidth=1.2, ls="--")

cbar = fig.colorbar(im, ax=ax, pad=0.01)
cbar.set_label(f"Spike rate per {BIN_MS:g} ms bin (Hz)")
cbar.outline.set_linewidth(0.8)

ax.set_title(
    f"All selected electrode channels spike-rate heatmap\n"
    f"inputs {ELECTRODE_CHANNEL_INDICES[0]}..{ELECTRODE_CHANNEL_INDICES[-1]} | "
    f"bin={BIN_MS:g} ms | cmap={HEATMAP_CMAP}"
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Selected electrode channel")

# Put ticks in local channel coordinates but label as original all_channels index/name occasionally.
tick_step = 8
yticks = np.arange(0.5, n_ch + 0.5, tick_step)
ytick_locs = np.arange(0, n_ch, tick_step)

ax.set_yticks(yticks[:len(ytick_locs)])
ax.set_yticklabels([
    f"{ELECTRODE_CHANNEL_INDICES[i]}:{channel_names[i]}"
    for i in ytick_locs
], fontsize=8)

ax.set_ylim(0, n_ch)
ax.grid(False)

plt.tight_layout()
plt.show()

Using electrode inputs: 15..142
n_channels = 128
fs = 30000.0 Hz
duration = 180.879 sec
trace units from CHANNEL_INDEX 60: uV
Plot window = (70.0, 75.0)
Plotting original all_channels[60] = elec2-66
Local electrode-channel index = 45
Materializing high-pass traces...
Estimating noise levels by MAD...


noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

Using matched_filtering template:
  E:\NHP_Cerebellum_Project_2025_RCP_Analysis_Git\RCP_analysis\config\waveform_templates\median_extremum_templates_norm_UA_PortB.npy
  template shape = (105,)
  ms_before = 1.5000


detect peaks (matched_filtering) (no parallelization):   0%|          | 0/181 [00:00<?, ?it/s]

Detected peaks before amplitude/SNR filtering: 369,962
Detection method used: matched_filtering
Detected peaks after amplitude/SNR filtering:  182,884
Mean channel rate after filtering: 7.90 Hz
Median channel rate after filtering: 4.27 Hz
Max channel rate after filtering: 91.75 Hz


C:\Users\culle\AppData\Local\Temp\ipykernel_17860\1909261128.py:437: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_local = cm.get_cmap(HEATMAP_CMAP).copy()
